A project that adds an “Ask” button to YouTube videos, powered by a RAG (Retrieval-Augmented Generation) pipeline built with LangChain. It pulls relevant context from video transcripts or related data and lets users ask questions about the content, returning concise, context-aware answers in real time. Please check [link](https://www.youtube.com/watch?v=USgyt6sHJeg) for creating API_TOKEN

---



In [ ]:
import os
os.environ["HUGGINGFACEHUB_API_TOKEN"] = "<please create your API token>"

## Install libraries


In [ ]:
!pip install -q requests==2.32.4 youtube-transcript-api langchain-community langchain-huggingface transformers huggingface-hub \
               faiss-cpu tiktoken python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.2/485.2 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 42.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 27.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 53.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 459.1/459.1 kB 31.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langgraph-prebuilt 1.0.10 requires langchain-core>=1.0.0, but you have langchain-core 0.3.84 which is incompatible.
langgraph 1.1.8 requires langchain-core<2,>=1.3.0, but you have langchain-core 0.3.84 which is incompatible.


In [ ]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
from youtube_transcript_api._errors import TranscriptsDisabled, NoTranscriptFound
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings, ChatHuggingFace, HuggingFaceEndpoint
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate

## Step 1a - Indexing (Document Ingestion)

In [ ]:
from youtube_transcript_api import YouTubeTranscriptApi

video_id = "rcpNFm_poQs"

try:
    ytt = YouTubeTranscriptApi()

    transcript_list = ytt.fetch(video_id)

    transcript = " ".join(chunk.text for chunk in transcript_list)

    print(transcript_list)

except Exception as e:
    print("Error:", e)

FetchedTranscript(snippets=[FetchedTranscriptSnippet(text='piece of code and welcome to this video.', start=3.88, duration=4.08), FetchedTranscriptSnippet(text='Now in this video, this video is going', start=6.24, duration=3.36), FetchedTranscriptSnippet(text='to be very special because I want to', start=7.96, duration=4.64), FetchedTranscriptSnippet(text='discuss about one more really amazing', start=9.6, duration=6.0), FetchedTranscriptSnippet(text='and this is going to be very popular', start=12.6, duration=5.88), FetchedTranscriptSnippet(text='you know, AI certification and', start=15.6, duration=5.28), FetchedTranscriptSnippet(text='this is from none other than Anthropic', start=18.48, duration=4.16), FetchedTranscriptSnippet(text='AI or Claude, whatever you want to call', start=20.88, duration=2.68), FetchedTranscriptSnippet(text='it.', start=22.64, duration=3.2), FetchedTranscriptSnippet(text='And this is Claude certified architect', start=23.56, duration=6.48), FetchedTranscrip

## Step 1b - Indexing (Text Splitting)

In [ ]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.create_documents([transcript])
len(chunks)

10

## Step 1c & 1d - Indexing (Embedding Generation and Storing in Vector Store)

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
# vector = embedding.embed_documents(documnet)  # Invoke the model with a prompt
vector_store = FAISS.from_documents(chunks, embedding)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
vector_store.index_to_docstore_id
vector_store.get_by_ids(['d6bca151-1092-4287-ba3e-17c95c21a331'])

[]

## Step 2 - Retrieval

In [ ]:
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 4})

In [ ]:
retriever

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x7a05f3478050>, search_kwargs={'k': 4})

In [ ]:
retriever.invoke('What is discussed in this video ?')

[Document(id='53327528-6ded-4d72-8042-a575ef33acef', metadata={}, page_content="piece of code and welcome to this video. Now in this video, this video is going to be very special because I want to discuss about one more really amazing and this is going to be very popular you know, AI certification and this is from none other than Anthropic AI or Claude, whatever you want to call it. And this is Claude certified architect guys and it has been recently launched. And um it has some common gotchas that you need to be keeping in mind. Um What do I mean by common gotchas is just some uh things that you need to keep in mind before appearing for this particular exam. First of all, the exam is exclusive for Anthropic partners. Okay? It just means that it is not open to individual you know, um subscribers or people. It is for Anthropic partners. For example, you are working at a company who is a partner to Anthropic and you're employed there, so you will be able to give this exam and giving this

## Step 3 - Augmentation

In [ ]:
prompt = PromptTemplate(
    template="""
      You are a helpful assistant.
      Answer ONLY from the provided transcript context.
      If the context is insufficient, just say you don't know.

      {context}
      Question: {question}
    """,
    input_variables = ['context', 'question']
)

## Step 4 - Generation

In [ ]:
question          = "What is discussed in this video ?"
retrieved_docs    = retriever.invoke(question)

In [ ]:
retrieved_docs

[Document(id='53327528-6ded-4d72-8042-a575ef33acef', metadata={}, page_content="piece of code and welcome to this video. Now in this video, this video is going to be very special because I want to discuss about one more really amazing and this is going to be very popular you know, AI certification and this is from none other than Anthropic AI or Claude, whatever you want to call it. And this is Claude certified architect guys and it has been recently launched. And um it has some common gotchas that you need to be keeping in mind. Um What do I mean by common gotchas is just some uh things that you need to keep in mind before appearing for this particular exam. First of all, the exam is exclusive for Anthropic partners. Okay? It just means that it is not open to individual you know, um subscribers or people. It is for Anthropic partners. For example, you are working at a company who is a partner to Anthropic and you're employed there, so you will be able to give this exam and giving this

In [ ]:
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
context_text

"piece of code and welcome to this video. Now in this video, this video is going to be very special because I want to discuss about one more really amazing and this is going to be very popular you know, AI certification and this is from none other than Anthropic AI or Claude, whatever you want to call it. And this is Claude certified architect guys and it has been recently launched. And um it has some common gotchas that you need to be keeping in mind. Um What do I mean by common gotchas is just some uh things that you need to keep in mind before appearing for this particular exam. First of all, the exam is exclusive for Anthropic partners. Okay? It just means that it is not open to individual you know, um subscribers or people. It is for Anthropic partners. For example, you are working at a company who is a partner to Anthropic and you're employed there, so you will be able to give this exam and giving this exam also comes in, you know, a little bit of a twist. So, first of all, it is

In [ ]:
final_prompt = prompt.invoke({"context": context_text, "question": question})
final_prompt

StringPromptValue(text="\n      You are a helpful assistant.\n      Answer ONLY from the provided transcript context.\n      If the context is insufficient, just say you don't know.\n\n      piece of code and welcome to this video. Now in this video, this video is going to be very special because I want to discuss about one more really amazing and this is going to be very popular you know, AI certification and this is from none other than Anthropic AI or Claude, whatever you want to call it. And this is Claude certified architect guys and it has been recently launched. And um it has some common gotchas that you need to be keeping in mind. Um What do I mean by common gotchas is just some uh things that you need to keep in mind before appearing for this particular exam. First of all, the exam is exclusive for Anthropic partners. Okay? It just means that it is not open to individual you know, um subscribers or people. It is for Anthropic partners. For example, you are working at a company

In [ ]:
llm = HuggingFaceEndpoint(repo_id="MiniMaxAI/MiniMax-M2.7", task="text-generation")
model = ChatHuggingFace(llm=llm)  # Initialize the HuggingFace Chat Model.

In [ ]:
answer = model.invoke(final_prompt)
print(answer.content)

The video introduces Anthropic AI’s new “Claude Certified Architect” certification, which is only available to employees of Anthropic partner companies. It covers the exam’s key details (120 minutes, proctored, results in about two days) and outlines the syllabus: agentic architecture & orchestration, tool design and MCP integration, context‑management techniques (e.g., using the `/compact` command), and reliability considerations. The presenter also shares common “gotchas” – such as registering with an official company email and noting that the exam is not open to individuals – and briefly mentions sponsorship from the partner company.


## Building a Chain

In [ ]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [ ]:
def format_docs(retrieved_docs):
  context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
  return context_text

In [ ]:
parallel_chain = RunnableParallel({
    'context': retriever | RunnableLambda(format_docs),
    'question': RunnablePassthrough()
})

In [ ]:
parallel_chain.invoke('What is the strategy explained in the video')

{'context': 'HIS STRATEGY STRATEGY IS SOUND AND I THINK IT\'S GOING TO BE EXTREMELY SUCCESSFUL. THE LAST AND ONLY COMPONENT NOT IN THE NEGOTIATION IS THE IRGC ITSELF. AND DON\'T FORGET, THIS IS THE ISLAMIC REVOLUTION. REVOLUTIONARY GUARD CORPS. THEY ARE THE DECISION MAKERS AND REAL POWER IN IRAN THE REAL ELEMENT SUPPRESSING ITS PEOPLE. AT THE END OF THE DAY, IF YOU REMOVE ALL THESE PROGRAMS THEY HAVE BUILT TO MAINTAIN POWER, THEN THEY BECOME AT RISK. AND SO THEY ARE BOTH CLOSELY TIED TO EACH OTHER. IF THEY WON\'T NEGOTIATE, WE TAKE THEM OFF THE STEP-BY-STEP. >> Brian: WHAT IS SO IMPORTANT IS, PEOPLE IN ACADEMICS ARE SPEAKING OUT OVER THE WEEKEND AND OTHERS, IT\'S NOT THE UAE AND SAUDI ARABIA RECONSIDERING THEIR RELATIONSHIP WITH US AND OUR BASES. THAT\'S JUST NOT TRUE. AND WE GET MORE AND MORE REPORTS THAT MBS SAYS SPECIFICALLY MR. PRESIDENT, FINISH THE JOB, I WILL BACK YOU UP FINANCIALLY. FINISH THE JOB. THE "WALL STREET JOURNAL" REPORTED LAST NIGHT THAT PRESIDENT TRUMP AND HIS ADVISE

In [ ]:
parser = StrOutputParser()

In [ ]:
main_chain = parallel_chain | prompt | model | RunnableLambda(lambda x: x.content) | parser

In [ ]:
main_chain.invoke('What is the strategy explained in the video')

'Based on the provided transcript, the strategy explained involves the following components:\n\n*   **Targeting the IRGC:** Specifically targeting the Islamic Revolutionary Guard Corps (IRGC)—who are described as the real power and decision-makers in Iran—using limited military strikes similar to what the Israelis have done. If the IRGC will not negotiate, the strategy is to take them off the step-by-step negotiation process.\n*   **Economic and Military Pressure:** Implementing a blockade and considering limited military strikes in Iran, as well as freezing $27 billion in Iranian funds. \n*   **Strict Demands:** Requiring Iran to stop enriching uranium and stop funding surrogates. \n*   **Refusing Negotiations Without Capitulation:** Walking away from negotiations if Iran does not capitulate to the demands ("he gave them a shot they didn\'t capitulate. We\'re done"). \n*   **Ultimate Goal:** Achieving enduring peace and stability in the region by establishing a government that renounc

## Miscellaneous

### Listing Available Hugging Face Models

In [ ]:
from huggingface_hub import HfApi

hf_api = HfApi()

# List models and then filter by 'text-generation' pipeline_tag
# We'll fetch a larger number of models and then filter
all_models = hf_api.list_models(limit=100) # Fetch up to 100 models, adjust limit as needed

text_generation_models = []
for model in all_models:
    if model.pipeline_tag == "text-generation":
        text_generation_models.append(model)
    if len(text_generation_models) >= 10: # Get top 10 text-generation models
        break

print("Top 10 models for text-generation:")
for model in text_generation_models:
    print(f"- {model.id}")

# You can also filter by other criteria, e.g., models frequently downloaded
# popular_models = hf_api.list_models(sort="downloads", direction=-1, limit=10)
# print("\nTop 10 most downloaded models:")
# for model in popular_models:
#     print(f"- {model.id}")

Top 10 models for text-generation:
- zai-org/GLM-5.1
- MiniMaxAI/MiniMax-M2.7
- LilaRest/gemma-4-31B-it-NVFP4-turbo
- Jiunsong/supergemma4-26b-uncensored-gguf-v2
- unsloth/GLM-5.1-GGUF
- nvidia/Gemma-4-31B-IT-NVFP4
- prism-ml/Bonsai-8B-gguf
- unsloth/MiniMax-M2.7-GGUF
- Jiunsong/supergemma4-26b-uncensored-mlx-4bit-v2
- douyamv/Gemma-4-31B-JANG_4M-CRACK-GGUF


In [ ]:
import importlib.metadata
print(importlib.metadata.version("youtube-transcript-api"))

# import requests

# video_id = "dQw4w9WgXcQ" # Changed to match the video_id in the previous cell
# url = f"https://www.youtube.com/api/timedtext?v={video_id}"

# res = requests.get(url)
# print(f"Status Code: {res.status_code}")
# print(f"Response Text (first 500 chars):\n{res.text[:500]}")

1.2.4


In [ ]:
video_id = "Gfr50f6ZBvo"

try:
    # Using a hardcoded transcript for demonstration purposes due to API issues
    transcript = "This is a sample text representing a YouTube video transcript. It will be used to demonstrate the text splitting functionality. We need enough text to create multiple chunks. Large language models (LLMs) are a type of artificial intelligence (AI) program that can recognize and generate text and other content, and answer questions. Large language models are trained on vast amounts of data and can be used for a variety of tasks, including natural language processing, language translation, and text generation. The field of AI is rapidly evolving, with new breakthroughs happening frequently. This example showcases how to prepare data for a Retrieval Augmented Generation (RAG) system, where documents are split into smaller, manageable chunks for efficient retrieval. The goal is to create a robust system that can answer questions based on the content of these documents."

    # Initialize the text splitter
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=200,
        chunk_overlap=50,
        length_function=len,
        is_separator_regex=False,
    )

    # Split the transcript into documents (chunks)
    documents = text_splitter.create_documents([transcript])

    print(f"Using hardcoded transcript for video_id: {video_id}")
    print(f"Number of document chunks created: {len(documents)}")

except Exception as e:
    print(f"An unexpected error occurred during text splitting: {e}")

Using hardcoded transcript for video_id: Gfr50f6ZBvo
Number of document chunks created: 6
